In [11]:
import pandas as pd
import numpy as np
import kagglehub
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

In [12]:
# 1. Download the dataset using your provided command
path = kagglehub.dataset_download("mabubakrsiddiq/eyesight-and-vision-health-synthetic-dataset")

In [13]:
# Find the actual CSV file inside the downloaded directory
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
file_path = os.path.join(path, csv_file)

# Load the dataset into a Pandas DataFrame
df_raw = pd.read_csv(file_path)

df_raw = df_raw.drop(columns=[c for c in ["Unnamed: 0", "id"] if c in df_raw.columns])

# Binary target: 1 = wears glasses, 0 = does not
df_raw["wear_glasses"] = (df_raw["glasses_number"] > 0).astype(int)

print(f"Dataset shape: {df_raw.shape}")
print(f"Class balance  — wears glasses: {df_raw['wear_glasses'].sum()}  |  no glasses: {(df_raw['wear_glasses'] == 0).sum()}")
df_raw.head()

Dataset shape: (10000, 12)
Class balance  — wears glasses: 5225  |  no glasses: 4775


,exercise_hours,mental_health_score,screen_time_hours,screen_brightness_avg,age,height_cm,outdoor_light_exposure_hours,night_mode_usage,screen_distance_cm,glasses_number,eye_health_score,wear_glasses
0,3.441116,50.112741,4.387540,68.531464,56,172.766324,1.821210,79.091607,33.408167,3,45.492089,1
1,7.494288,66.181801,9.596943,54.460165,19,180.683155,0.455726,90.535187,54.127821,0,74.610049,0
2,2.733887,69.674360,12.272036,74.334277,76,185.319068,0.301454,11.488773,50.769790,0,52.643897,0
3,8.122516,70.996764,8.820065,56.450697,65,166.770667,1.226576,83.373275,51.267787,2,65.963707,1
4,1.769984,50.017834,10.962927,87.181496,25,165.203603,0.521827,89.394952,54.595573,0,66.109601,0


In [14]:
# Feature Engineering

FEATURE_CONFIG = {
    "near_work_intensity":    True,
    "light_dose_near":        True,
    "night_mode_ratio":       True,
    "mh_age_interaction":     True,
    "screen_bin":             True,
    "age_squared":            True,
}

def engineer_features(data: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = data.copy()
    if config.get("age_squared"):
        df["age"] = df["age"] ** 2
    if config.get("near_work_intensity"):
        df["near_work_intensity"] = df["screen_time_hours"] * (1 / df["screen_distance_cm"])
    if config.get("light_dose_near"):
        df["light_dose_near"] = df["screen_time_hours"] * df["screen_brightness_avg"]
    if config.get("night_mode_ratio"):
        df["night_mode_ratio"] = df["night_mode_usage"] / (df["screen_time_hours"] + 0.1)
    if config.get("mh_age_interaction"):
        df["mh_age_interaction"] = df["mental_health_score"] * df["age"]
    if config.get("screen_bin"):
        df["screen_bin"] = pd.cut(
            df["screen_time_hours"],
            bins=[0, 1, 4, 8, np.inf],
            labels=["<=1h", "1-4h", "4-8h", ">8h"]
        )
    return df

df = engineer_features(df_raw, FEATURE_CONFIG)
print(f"Columns after feature engineering: {list(df.columns)}")

Columns after feature engineering: ['exercise_hours', 'mental_health_score', 'screen_time_hours', 'screen_brightness_avg', 'age', 'height_cm', 'outdoor_light_exposure_hours', 'night_mode_usage', 'screen_distance_cm', 'glasses_number', 'eye_health_score', 'wear_glasses', 'near_work_intensity', 'light_dose_near', 'night_mode_ratio', 'mh_age_interaction', 'screen_bin']


In [15]:
# Shared Pipeline Helpers

# Columns that must never enter a model as features
drop_columns = ["wear_glasses", "glasses_number", "eye_health_score", "height_cm"]

random_state = 42
test_split = 0.2


def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    cat_cols = [c for c in X.columns if X[c].dtype.name in ("category", "object")]
    num_cols = [c for c in X.columns if c not in cat_cols]
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", "passthrough", num_cols),
        ],
        remainder="drop",
    )


def build_pipeline(clf, X: pd.DataFrame, scale: bool = False) -> Pipeline:
    """Wrap a classifier in a preprocessing + optional scaling pipeline."""
    preprocessor = make_preprocessor(X)
    steps = [("preprocess", preprocessor)]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("clf", clf))
    return Pipeline(steps=steps)


def prepare_split(data: pd.DataFrame, extra_drop: list[str] = None):
    """Split into train/test; drop leakage columns automatically."""
    drop = drop_columns + (extra_drop or [])
    y = data["wear_glasses"]
    X = data.drop(columns=[c for c in drop if c in data.columns])
    return train_test_split(X, y, test_size=test_split, random_state=random_state, stratify=y)


def model_metrics(name: str, y_true, y_pred, y_proba) -> dict:
    return {
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_true, y_pred),  4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_true, y_pred,    zero_division=0), 4),
        "F1":        round(f1_score(y_true, y_pred,        zero_division=0), 4),
        "ROC_AUC":   round(roc_auc_score(y_true, y_proba), 4),
    }


def run_model(name: str, clf, X_train, X_test, y_train, y_test, scale: bool = False) -> tuple[dict, object]:
    pipe = build_pipeline(clf, X_train, scale=scale)
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    metrics = model_metrics(name, y_test, y_pred, y_proba)
    # print(f"[{name}]  Accuracy={metrics['Accuracy']}  F1={metrics['F1']}  ROC_AUC={metrics['ROC_AUC']}")
    return metrics, pipe

# Supply new observations here
# Keys must match the raw feature columns (before feature engineering).
# Feature engineering is applied automatically inside predict_for().

def predict_for(raw_input: dict) -> int:
    row = pd.DataFrame([raw_input])
    row = engineer_features(row, FEATURE_CONFIG)
    # Drop any leakage columns that may accidentally be present
    row = row.drop(columns=[c for c in drop_columns if c in row.columns], errors="ignore")
    # Align columns to training set
    row = row.reindex(columns=X_train.columns, fill_value=0)
    return int(best_pipeline.predict(row)[0])

In [16]:
# Model Training
# Each entry:  (display_name, classifier, needs_scaling)
# Add / remove / modify entries here to experiment with new models.

# Class-imbalance weight for XGBoost
n_pos = df["wear_glasses"].sum()
n_neg = (df["wear_glasses"] == 0).sum()
scale_pos_weight_calc = n_neg / n_pos

MODEL_REGISTRY = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=random_state), True),
    ("Decision Tree", DecisionTreeClassifier(max_depth=5, random_state=random_state), False),
    ("Random Forest", RandomForestClassifier(n_estimators=300, random_state=random_state, n_jobs=-1), False),
    ("XGBoost", XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight_calc,
            n_jobs=-1,
            random_state=random_state,
        ), False),]

X_train, X_test, y_train, y_test = prepare_split(df)

results  = []   # metrics dicts
trained  = {}   # name → fitted pipeline

for name, clf, scale in MODEL_REGISTRY:
    metrics, pipe = run_model(name, clf, X_train, X_test, y_train, y_test, scale=scale)
    results.append(metrics)
    trained[name] = pipe

In [17]:
# Model Comparison
comparison_df = (
    pd.DataFrame(results)
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)
comparison_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.6785,0.7166,0.6364,0.6741,0.7515
1,XGBoost,0.6660,0.6970,0.6383,0.6663,0.7381
2,Decision Tree,0.6660,0.7033,0.6239,0.6613,0.7362
3,Random Forest,0.6745,0.6898,0.6852,0.6875,0.7362


In [18]:
# Prediction with Best Model
best_model_name = comparison_df.iloc[0]["Model"]
best_pipeline = trained[best_model_name]
print(f"Using model: {best_model_name}")

Using model: Logistic Regression
Prediction: 0  (no glasses)


In [ ]:
# Example call
sample = {
    "exercise_hours":              3.5,
    "mental_health_score":         65.0,
    "screen_time_hours":           8.0,
    "screen_brightness_avg":       70.0,
    "age":                         30,
    "outdoor_light_exposure_hours": 1.5,
    "night_mode_usage":            50.0,
    "screen_distance_cm":          50.0,
}

prediction = predict_for(sample)
print(f"Prediction: {prediction}  ({'wears glasses' if prediction == 1 else 'no glasses'})")